# Met One AIO reader and quicklooks — GAIL

Read a monthly `AIO_GAIL_YYYYMM.dat` file, select one UTC day, inspect the normalized product, and create the standard overview and wind-rose quicklooks.

In [ ]:
from pathlib import Path
import os
import sys

GV_HOME = Path(
    os.environ.get("GV_TOOLS_HOME", Path.home() / "Desktop" / "Work" / "GV Tools")
).expanduser().resolve()
PACKAGE = GV_HOME / "gv_tools"
SOURCE = PACKAGE / "src"
if SOURCE.is_dir() and str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
if not PACKAGE.is_dir():
    raise FileNotFoundError(f"GV Tools project not found at {PACKAGE}. Set GV_TOOLS_HOME.")

INPUT_DIR = Path(os.environ.get("GV_TOOLS_METONE_INPUT", "/Volumes/TBW/distro/aio")).expanduser()
FILES = [INPUT_DIR / "AIO_GAIL_202501.dat"]
OUTPUT_DIR = Path(os.environ.get("GV_TOOLS_OUTPUT", GV_HOME / "Output")).expanduser()
DATE = "2025-01-15"
INPUT_DIR, FILES, DATE, OUTPUT_DIR


In [ ]:
import gv_tools

if gv_tools.__version__ != "0.29.4":
    raise RuntimeError(
        f"This notebook requires GV Tools 0.29.4, but the Jupyter kernel has "
        f"{gv_tools.__version__} loaded. Restart the kernel, then Run All cells."
    )

## Discover and decode

In [ ]:
available_dates = gv_tools.io.discover_aio(FILES)
print(f"{len(available_dates)} available days:", available_dates[:3], "...", available_dates[-3:])

raw = gv_tools.io.read_aio_raw(FILES)
print("Source files:", raw.source_files)
print("Rejected rows:", raw.rejected_rows)
print("Duplicate rows:", raw.duplicate_rows)
raw.observations.head()


## Normalize and optionally write the daily product

In [ ]:
aio = gv_tools.io.read_aio(FILES)
aio


In [ ]:
aio.to_dataframe().describe()

In [ ]:
created = gv_tools.io.write_product(aio, OUTPUT_DIR, formats=('csv',), day=DATE)
created

## Standard quicklooks

In [ ]:
overview_dir = gv_tools.graph.plot_directory(OUTPUT_DIR, 'MetOne_AIO_Overview', DATE)
overview_path = overview_dir / f'GAIL_AIO_{DATE.replace("-", "")}_overview.png'
gv_tools.graph.plot_aio_quicklook(aio, overview_path, title=f'GAIL Met One AIO — {DATE}')

In [ ]:
wind_dir = gv_tools.graph.plot_directory(OUTPUT_DIR, 'MetOne_AIO_WindRose', DATE)
wind_path = wind_dir / f'GAIL_AIO_{DATE.replace("-", "")}_windrose.png'
figure, axis, frequencies = gv_tools.graph.plot_aio_wind_rose(
    aio, wind_path, title=f'GAIL Met One AIO wind rose — {DATE}'
)
frequencies.sum(), wind_path